# Setup NemoClaw

This notebook is meant to run **on the host itself** — a [Brev](https://brev.nvidia.com) instance or any other Linux host. It walks through the host-side flow for **NemoClaw**: configure or reuse a sandbox, pin the tested Docker package versions, install NemoClaw and onboard the sandbox with your chosen agent harness (OpenClaw or Hermes), apply the VSS policy, install the VSS skills, upload the workspace bootstrap docs, optionally register an HTTPS VSS Orchestrator MCP path (only when `ORCHESTRATOR_ENABLE_HTTPS` is `True`; the default HTTP path is started later by `deploy_vss_orchestrator.ipynb` and needs no registration here), configure optional OpenClaw webhooks, then optionally verify the live sandbox, active policy, webhooks, and installed workspace docs.

A few steps are Brev-specific — the notebook reads secure-link FQDNs from `BREV_ENVIRONMENT_CONTEXT_PATH` (default `/etc/brev/environment-context.json`) and the generated remote UI link — and are called out where they apply. On other platforms, install the prerequisites yourself and reach the agent UI over your own networking (e.g. the SSH tunnel shown in section 3.7).

Once NemoClaw is up (and you have opened the Agent UI in section 3.7), continue with the companion notebook **`deploy_vss_orchestrator.ipynb`** to prepare the host, start the VSS Orchestrator MCP server, and deploy/manage VSS from the agent UI.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere or the host has multiple clones.
- If you are re-onboarding a host that previously ran an older OpenShell gateway, follow the NemoClaw sandbox lifecycle upgrade guidance: <https://docs.nvidia.com/nemoclaw/manage-sandboxes/lifecycle>.

**What this notebook covers**

- Choose an agent model provider (which sets the required API key) and set notebook options.
- Run preflight checks for the local repo, scripts, policy file, and host prerequisites, then pin Docker to the tested package versions.
- Create or reuse the NemoClaw sandbox, configure the chosen agent model provider, apply the VSS policy, install VSS skills, upload workspace docs, optionally register HTTPS VSS Orchestrator MCP access when enabled, and bake the agent UI origin via `CHAT_UI_URL` at onboard.
- Open the Agent UI (OpenClaw shown as an example); the companion notebook then verifies the agent and deploys VSS.
- *(Optional)* Verify the live sandbox state, active policy metadata, OpenClaw webhooks (if enabled), and installed workspace docs.

**Security:** prefer `NVIDIA_API_KEY` from environment variables or your platform's secret store (e.g. Brev secrets). Do **not** commit notebook outputs that contain credentials or live access tokens.


## 1. Settings

Configure the notebook in two parts:

1. **Initialize provider variables** — seed the agent model-provider variables before you choose a provider.
2. **Choose ONE agent model provider** — pick exactly one of (a) a SOTA cloud model, (b) a locally-hosted OpenAI-compatible model, or (c) a model from build.nvidia.com.

> Run section 1.1, then run **exactly one** of the (a) / (b) / (c) cells. Section 1.3 holds advanced defaults you can usually leave alone.

<span style="color:red"><strong>Important:</strong> set at least one of <code>NVIDIA_API_KEY</code> / <code>COMPATIBLE_API_KEY</code> via the provider cell you pick. Credentials can also come from the environment or your platform's secret store (e.g. Brev secrets).</span>

### 1.1 Initialize agent model-provider variables

Run this cell first. It seeds the four agent model-provider variables to empty so that whichever **one** of the (a)/(b)/(c) cells you run in section 1.2 works on its own.


In [ ]:
# ================== Agent model vars (set by ONE of (a)/(b)/(c) in 1.2) ==================
NVIDIA_API_KEY = NEMOCLAW_ENDPOINT_URL = NEMOCLAW_MODEL = COMPATIBLE_API_KEY = ""

### 1.2 Choose ONE agent model provider

Run **exactly one** of the cells below. 

| Option | When to use | Sets |
|---|---|---|
| **(a) SOTA cloud model** | Best agent quality (Claude Opus, GPT-5, …) via any OpenAI-compatible cloud API. | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(b) Local OpenAI-compatible model** | Self-hosted on this box or LAN | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(c) build.nvidia.com NVIDIA-hosted model** | Default path — uses NVIDIA's hosted Nemotron via `integrate.api.nvidia.com`. | `NVIDIA_API_KEY`, optionally `NEMOCLAW_MODEL` |


#### (a) SOTA cloud model — *recommended for best agent quality*



In [ ]:
# (a) SOTA cloud model — fill in these three values, then run.
#

NEMOCLAW_ENDPOINT_URL = ""  # OpenAI-compatible base URL, e.g. "https://api.anthropic.com/v1/"
NEMOCLAW_MODEL        = ""  # Model id at that endpoint, e.g. "claude-opus-4-6"
COMPATIBLE_API_KEY    = ""  # Bearer token (sk-ant-..., sk-proj-..., etc.)

#### (b) Local OpenAI-compatible model — *self-hosted / air-gapped*

Point the agent at a local OpenAI-compatible server you've already started on this host or your LAN. For example, a downloadable NIM from build.nvidia.com


> `COMPATIBLE_API_KEY` is technically required by the installer (it errors if blank), but most local servers ignore the value — any non-empty placeholder works.
> From inside the NemoClaw sandbox, the host is reachable as `host.openshell.internal`. If your server binds only to `127.0.0.1` on the host, rebind it to `0.0.0.0` — the sandbox cannot reach a loopback-only socket via `host.openshell.internal`.

In [ ]:
# (b) Local OpenAI-compatible model — fill in to match your local server, then run.
NEMOCLAW_ENDPOINT_URL = "http://host.openshell.internal:8000/v1"
NEMOCLAW_MODEL        = "nvidia/nemotron-3-super-120b-a12b"  # the id your local server reports
COMPATIBLE_API_KEY    = "EMPTY"                              # most local servers ignore this; must be non-empty


#### (c) build.nvidia.com NVIDIA-hosted model — *default, zero-setup*

Uses NVIDIA's hosted model catalog via `integrate.api.nvidia.com`. Only `NVIDIA_API_KEY` is required; leave `NEMOCLAW_MODEL` blank to use the NemoClaw default.

> Get a key at <https://build.nvidia.com> (format `nvapi-...`). To use a different hosted model, set `NEMOCLAW_MODEL` to its build.nvidia.com id (e.g. `nvidia/llama-3.3-nemotron-super-49b-v1.5`).


In [ ]:
# (c) build.nvidia.com — set NVIDIA_API_KEY; clear the custom-endpoint vars so the installer picks NEMOCLAW_PROVIDER=build.
NVIDIA_API_KEY = ""  # "nvapi-..." from https://build.nvidia.com
NEMOCLAW_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"  # leave blank to use the NemoClaw default,, or set a build.nvidia.com model id

### 1.3 Advanced settings (defaults — usually leave alone)

Pinned installer ref, sandbox agent harness, the inference API proxy toggle, agent webhook plumbing, and the skill-install concurrency. Set `AGENT_RUNTIME` to a key in `AGENT_HARNESS_PROFILES` (`openclaw` or `hermes` today); override via the `AGENT_RUNTIME` env var. To support a new harness, add a profile entry — downstream cells consume the derived fields. `AGENT_DASHBOARD_PORT` is derived (default `18789`; override via `NEMOCLAW_DASHBOARD_PORT`). Hermes webhook port defaults to `8644` (`AGENT_WEBHOOK_PORT`). `BREV_ENVIRONMENT_CONTEXT_PATH` is derived (default `/etc/brev/environment-context.json`) — override via that env var only if the context file lives elsewhere. `SKILL_INSTALL_WORKERS` (default `4`) is how many `skill install` calls section 3.3 runs at once.

`ORCHESTRATOR_ENABLE_HTTPS` gates section 3.5: when `False` (the default), that cell is a no-op because the agent uses the locally deployed HTTP orchestrator MCP started by `deploy_vss_orchestrator.ipynb` (no sandbox `mcp add` needed); when `True`, it registers the HTTPS orchestrator MCP URL with the sandbox. The server itself is configured and started in `deploy_vss_orchestrator.ipynb`. Set the same value in both notebooks (or override via the `ORCHESTRATOR_ENABLE_HTTPS` env var).


In [ ]:
import os
import subprocess
import time
from pathlib import Path


# ================== Default Nemoclaw settings ==================
# NemoClaw installer pin. Must ship the public sandbox-first lifecycle
# commands used below (`{cli} {sandbox} policy-add|skill install|mcp|…`).
NEMOCLAW_INSTALL_REF = "v0.0.109"
# Sandbox agent harness: "openclaw" or "hermes". Override via the AGENT_RUNTIME env var.
# To add a new harness, extend AGENT_HARNESS_PROFILES below — downstream cells read the profile.
AGENT_RUNTIME = "openclaw"
# Enable SSRF detection for NEMOCLAW_ENDPOINT_URL and host proxy startup.
NEMOCLAW_INFERENCE_PROXY = True
AGENT_HOOKS_ENABLED = True     # enable agent inbound webhooks when the harness supports them
AGENT_HOOKS_PATH = "/hooks"     # OpenClaw hooks path (ignored for Hermes)
AGENT_WEBHOOK_PORT = 8644       # Hermes webhook adapter port (ignored for OpenClaw)
# Must match ORCHESTRATOR_ENABLE_HTTPS in deploy_vss_orchestrator.ipynb.
ORCHESTRATOR_ENABLE_HTTPS = False
SKILL_INSTALL_WORKERS = 4       # concurrent `skill install` calls in 3.3; 1 installs serially


# ================== Derived (no need to touch) ==================

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
# Brev context JSON (secure-link FQDNs + environment id). Override via env if needed.
BREV_ENVIRONMENT_CONTEXT_PATH = os.environ.get("BREV_ENVIRONMENT_CONTEXT_PATH", "/etc/brev/environment-context.json").strip()
os.environ["BREV_ENVIRONMENT_CONTEXT_PATH"] = BREV_ENVIRONMENT_CONTEXT_PATH
# Agent UI port (OpenClaw / Hermes). Override via NEMOCLAW_DASHBOARD_PORT if needed.
AGENT_DASHBOARD_PORT = int(os.environ.get("NEMOCLAW_DASHBOARD_PORT", "18789") or "18789")
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
NEMOCLAW_REPO_DIR = Path(os.environ.get("NEMOCLAW_REPO_DIR", HOME_DIR / "NemoClaw")).resolve()
INFERENCE_API_PROXY_PORT = int(os.environ.get("INFERENCE_API_PROXY_PORT", "18080"))
AGENT_RUNTIME = (os.environ.get("AGENT_RUNTIME") or AGENT_RUNTIME).strip().lower()

# Fill `{cli}` here once the harness is chosen; later cells still
# `.format(sandbox=..., path=..., …)` the remaining placeholders.
def _with_cli(tmpl: str, cli: str) -> str:
    return tmpl.replace("{cli}", cli)


_AGENT_SANDBOX_CMDS = {
    "policy_add_cmd": "{cli} {sandbox} policy-add --from-file {path} --yes",
    "skill_install_cmd": "{cli} {sandbox} skill install {skill}",
    "upload_cmd": "{cli} {sandbox} upload {doc} {dest}/",
    "mcp_status_cmd": "{cli} {sandbox} mcp status {server}",
    "mcp_add_cmd": "{cli} {sandbox} mcp add {server} --url {url}",
    "mcp_remove_cmd": "{cli} {sandbox} mcp remove {server}",
    "config_set_cmd": "{cli} {sandbox} config set --key {key} --value {value} --config-accept-new-path",
    "gateway_restart_cmd": "{cli} {sandbox} gateway restart",
    "recover_cmd": "{cli} {sandbox} recover",
    "gateway_token_cmd": "{cli} {sandbox} gateway-token --quiet",
    "dashboard_url_cmd": "{cli} {sandbox} dashboard-url --quiet",
    "connect_cmd": "{cli} {sandbox} connect",
}

# Per-harness differences only (cli, onboard, paths, UI, verify).
AGENT_HARNESS_PROFILES = {
    "openclaw": {
        "label": "NemoClaw",
        "cli": "nemoclaw",
        "onboard_args": "--non-interactive --agent openclaw",
        "retry_onboard_on_fail": True,
        "workspace_remote_dir": "/sandbox/.openclaw/workspace",
        "ui_mode": "gateway_token",  # control UI via #token= from gateway-token
        "verify_kind": "openclaw_workspace",
    },
    "hermes": {
        "label": "NemoHermes",
        "cli": "nemohermes",
        "onboard_args": "--non-interactive",
        "retry_onboard_on_fail": True,
        "workspace_remote_dir": "/sandbox",
        "ui_mode": "dashboard_url",  # plain URL; gateway-token is the :8642 API bearer
        "verify_kind": "sandbox_docs",
        "verify_cmds": (
            ("{label} top-level docs", "ls -1 {workspace_remote_dir}/*.md 2>/dev/null || true"),
            (
                "VSS Orchestrator MCP host alias",
                "curl -s -o /dev/null --max-time 5 {scheme}://{host_alias}:{mcp_port}/ && echo host alias reachable || true",
            ),
        ),
    },
}
AGENT_HARNESS = AGENT_HARNESS_PROFILES.get(AGENT_RUNTIME)
if AGENT_HARNESS is None:
    known = ", ".join(sorted(AGENT_HARNESS_PROFILES))
    raise KeyError(f"Unknown AGENT_RUNTIME={AGENT_RUNTIME!r}; known harnesses: {known}")

AGENT_LABEL = AGENT_HARNESS["label"]
AGENT_CLI = AGENT_HARNESS["cli"]
_onboard_args = AGENT_HARNESS["onboard_args"].strip()
AGENT_ONBOARD_CMD = f"{AGENT_CLI} onboard {_onboard_args}"
AGENT_ONBOARD_FRESH_CMD = f"{AGENT_CLI} onboard --fresh {_onboard_args}"
AGENT_RETRY_ONBOARD_ON_FAIL = bool(AGENT_HARNESS["retry_onboard_on_fail"])
WORKSPACE_REMOTE_DIR = AGENT_HARNESS["workspace_remote_dir"]
_ui_mode = AGENT_HARNESS.get("ui_mode") or "dashboard_url"
AGENT_UI_USES_GATEWAY_TOKEN = _ui_mode == "gateway_token"
AGENT_POLICY_ADD_CMD = _with_cli(_AGENT_SANDBOX_CMDS["policy_add_cmd"], AGENT_CLI)
AGENT_SKILL_INSTALL_CMD = _with_cli(_AGENT_SANDBOX_CMDS["skill_install_cmd"], AGENT_CLI)
AGENT_UPLOAD_CMD = _with_cli(_AGENT_SANDBOX_CMDS["upload_cmd"], AGENT_CLI)
AGENT_MCP_STATUS_CMD = _with_cli(_AGENT_SANDBOX_CMDS["mcp_status_cmd"], AGENT_CLI)
AGENT_MCP_ADD_CMD = _with_cli(_AGENT_SANDBOX_CMDS["mcp_add_cmd"], AGENT_CLI)
AGENT_MCP_REMOVE_CMD = _with_cli(_AGENT_SANDBOX_CMDS["mcp_remove_cmd"], AGENT_CLI)
AGENT_CONFIG_SET_CMD = _with_cli(_AGENT_SANDBOX_CMDS["config_set_cmd"], AGENT_CLI)
AGENT_GATEWAY_RESTART_CMD = _with_cli(_AGENT_SANDBOX_CMDS["gateway_restart_cmd"], AGENT_CLI)
AGENT_RECOVER_CMD = _with_cli(_AGENT_SANDBOX_CMDS["recover_cmd"], AGENT_CLI)
AGENT_CONNECT_CMD = _with_cli(_AGENT_SANDBOX_CMDS["connect_cmd"], AGENT_CLI)
AGENT_GATEWAY_TOKEN_CMD = (
    _with_cli(_AGENT_SANDBOX_CMDS["gateway_token_cmd"], AGENT_CLI)
    if AGENT_UI_USES_GATEWAY_TOKEN
    else None
)
AGENT_DASHBOARD_URL_CMD = (
    _with_cli(_AGENT_SANDBOX_CMDS["dashboard_url_cmd"], AGENT_CLI)
    if _ui_mode == "dashboard_url"
    else None
)
AGENT_VERIFY_KIND = AGENT_HARNESS["verify_kind"]
AGENT_VERIFY_CMDS = AGENT_HARNESS.get("verify_cmds") or ()


# `gateway restart` and `recover` can exit non-zero with SUPERVISOR_UNAVAILABLE
# when their managed-control handshake gives up before the replacement gateway
# finishes booting, so ask the gateway itself before treating that as fatal.
def agent_gateway_healthy(timeout_s: int = 90, poll_s: int = 3, attempt_s: int = 15) -> bool:
    health_url = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}/health"
    # curl already reports 000 when it cannot connect, so no `||` fallback here.
    probe = f'curl -s -o /dev/null -w "%{{http_code}}" --max-time 5 {health_url}'
    deadline = time.monotonic() + timeout_s
    while True:
        try:
            # `sandbox exec` can wedge before it ever spawns curl, so each attempt
            # needs its own ceiling for the deadline below to mean anything.
            probed = subprocess.run(
                ["openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--", "sh", "-c", probe],
                capture_output=True,
                text=True,
                timeout=attempt_s,
            )
            code = (probed.stdout or "").strip()
        except subprocess.TimeoutExpired:
            code = ""
        # 401/403 still prove the gateway is listening behind its token.
        if code.startswith("2") or code in ("401", "403"):
            return True
        if time.monotonic() >= deadline:
            return False
        time.sleep(poll_s)


DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
NEMOCLAW_PROVIDER = "custom" if NEMOCLAW_ENDPOINT_URL else "build"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
SKILLS_DIR = VSS_REPO_DIR / "skills"
WORKSPACE_VARIANT = os.environ.get("AGENT_PLUGIN_VARIANT", "nemoclaw").strip() or "nemoclaw"
WORKSPACE_DIR = (VSS_REPO_DIR / ".openclaw" / "workspace").resolve()
SANDBOX_CONFIG_PATH = "/sandbox/.openclaw/openclaw.json"
AGENT_HOOKS_TOKEN = subprocess.check_output(["openssl", "rand", "-hex", "32"], text=True).strip() if AGENT_HOOKS_ENABLED else ""
INFERENCE_API_PROXY_PATH = DEPLOY_SCRIPTS_DIR / "nemoclaw" / "inference-api-proxy.py"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
BREV_UTIL_PATH = VSS_REPO_DIR / "services" / "agent" / "packages" / "vss_agents" / "src" / "vss_agents" / "orchestrator" / "brev_util.py"
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
HOST_INTERNAL_ALIAS = os.environ.get("HOST_INTERNAL_ALIAS", "host.openshell.internal").strip()
ORCHESTRATOR_ENABLE_HTTPS = (os.environ.get("ORCHESTRATOR_ENABLE_HTTPS", str(ORCHESTRATOR_ENABLE_HTTPS)).strip().lower() == "true")
MCP_SCHEME = "https" if ORCHESTRATOR_ENABLE_HTTPS else "http"
ORCHESTRATOR_MCP_SERVER = os.environ.get("ORCHESTRATOR_MCP_SERVER", "vss_orchestrator").strip() or "vss_orchestrator"
ORCHESTRATOR_MCP_URL = f"{MCP_SCHEME}://{HOST_INTERNAL_ALIAS}:{MCP_PORT}/mcp"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo").strip()
NEMOCLAW_INSTALL_REF = os.environ.get("NEMOCLAW_INSTALL_REF", NEMOCLAW_INSTALL_REF).strip()
AGENT_WEBHOOK_PORT = int(os.environ.get("AGENT_WEBHOOK_PORT", str(AGENT_WEBHOOK_PORT)) or "8644")
SKILL_INSTALL_WORKERS = max(1, int(os.environ.get("SKILL_INSTALL_WORKERS", str(SKILL_INSTALL_WORKERS))))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("NEMOCLAW_INSTALL_REF:", NEMOCLAW_INSTALL_REF)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_REPO_DIR:", NEMOCLAW_REPO_DIR)
print("POLICY_PATH:", POLICY_PATH)
print("SKILLS_DIR:", SKILLS_DIR)
print("SKILL_INSTALL_WORKERS:", SKILL_INSTALL_WORKERS)
print("WORKSPACE_DIR:", WORKSPACE_DIR)
print("WORKSPACE_REMOTE_DIR:", WORKSPACE_REMOTE_DIR)
print("INFERENCE_API_PROXY_PATH:", INFERENCE_API_PROXY_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("BREV_UTIL_PATH:", BREV_UTIL_PATH)
print("AGENT_RUNTIME:", AGENT_RUNTIME)
print("AGENT_LABEL:", AGENT_LABEL)
print("AGENT_CLI:", AGENT_CLI)
print("BREV_ENVIRONMENT_CONTEXT_PATH:", BREV_ENVIRONMENT_CONTEXT_PATH)
print("AGENT_DASHBOARD_PORT:", AGENT_DASHBOARD_PORT)
print("WORKSPACE_VARIANT:", WORKSPACE_VARIANT)
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("ORCHESTRATOR_MCP_SERVER:", ORCHESTRATOR_MCP_SERVER)
print("ORCHESTRATOR_MCP_URL:", ORCHESTRATOR_MCP_URL)
print("ORCHESTRATOR_ENABLE_HTTPS:", ORCHESTRATOR_ENABLE_HTTPS)
print("NEMOCLAW_PROVIDER:", NEMOCLAW_PROVIDER)
print("NEMOCLAW_INFERENCE_PROXY:", NEMOCLAW_INFERENCE_PROXY)
if NEMOCLAW_ENDPOINT_URL:
    print("NEMOCLAW_ENDPOINT_URL:", NEMOCLAW_ENDPOINT_URL)
    print("NEMOCLAW_MODEL:", NEMOCLAW_MODEL)
    print("COMPATIBLE_API_KEY set:", bool(COMPATIBLE_API_KEY))
print("Agent hooks enabled:", AGENT_HOOKS_ENABLED)
if AGENT_HOOKS_ENABLED:
    print("Agent hooks token set:", bool(AGENT_HOOKS_TOKEN))
    if AGENT_RUNTIME == "openclaw":
        print("Agent hooks path:", AGENT_HOOKS_PATH)
    elif AGENT_RUNTIME == "hermes":
        print("Agent webhook port:", AGENT_WEBHOOK_PORT)
print("NVIDIA_API_KEY set:", bool(NVIDIA_API_KEY))


## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the host.


In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

brev_util_spec = importlib.util.spec_from_file_location("vss_brev_util", BREV_UTIL_PATH)
if brev_util_spec is None or brev_util_spec.loader is None:
    raise ImportError(f"Could not load brev_util from {BREV_UTIL_PATH}")
brev_util = importlib.util.module_from_spec(brev_util_spec)
brev_util_spec.loader.exec_module(brev_util)

# Needed later: 3.1 (UI origin) and 3.7 (gateway container lookup).
brev_environment_id = brev_util.brev_environment_id
brev_secure_link_fqdn = brev_util.brev_secure_link_fqdn
resolve_openshell_gateway_container = orchestrator_mcp_helper.resolve_openshell_gateway_container


def agent_provider_configured() -> bool:
    if NEMOCLAW_PROVIDER == "build":
        return bool(NVIDIA_API_KEY)
    if NEMOCLAW_PROVIDER == "custom":
        return bool(NEMOCLAW_ENDPOINT_URL and NEMOCLAW_MODEL and COMPATIBLE_API_KEY)
    return False


required_checks = {
    "Agent model provider configured": agent_provider_configured(),
    "NEMOCLAW_INSTALL_REF set": bool(NEMOCLAW_INSTALL_REF),
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "workspace bootstrap dir": WORKSPACE_DIR.is_dir(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "brev_util.py": BREV_UTIL_PATH.is_file(),
    # host commands
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
}

if NEMOCLAW_PROVIDER == "custom":
    required_checks["NEMOCLAW_ENDPOINT_URL set"] = bool(NEMOCLAW_ENDPOINT_URL)
    required_checks["NEMOCLAW_MODEL set"] = bool(NEMOCLAW_MODEL)
    required_checks["COMPATIBLE_API_KEY set"] = bool(COMPATIBLE_API_KEY)
elif NEMOCLAW_PROVIDER == "build":
    required_checks["NVIDIA_API_KEY set (build provider)"] = bool(NVIDIA_API_KEY)

if AGENT_HOOKS_ENABLED:
    required_checks["AGENT_HOOKS_TOKEN set"] = bool(AGENT_HOOKS_TOKEN)
    if AGENT_RUNTIME == "openclaw":
        required_checks["AGENT_HOOKS_PATH set"] = bool(AGENT_HOOKS_PATH)
    elif AGENT_RUNTIME == "hermes":
        required_checks["AGENT_WEBHOOK_PORT set"] = bool(AGENT_WEBHOOK_PORT)

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")


### 2.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**) **before** section 3 brings up the NemoClaw sandbox — a docker-ce downgrade restarts dockerd and would disrupt live sandbox containers if it ran later in the notebook. `apt-mark hold` prevents drift back to newer versions for the rest of the deployment.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** the version downgrade is **skipped** — re-pinning to an exact version the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64) would fail with *version not found* for no benefit. The packages are still `apt-mark hold`-ed at their current versions so the box can't drift past the tested range mid-run. Safe to re-run.

In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the VSS deploy profiles are tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range -- keep in sync with the VSS launchable prereq check.
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 3. Install and Configure NemoClaw for VSS skills

Sections 3.1–3.6 install and configure the sandbox using canonical NemoClaw / OpenShell commands. Run the cells in order — each step is idempotent and safe to re-run. Section 3.7 opens the Agent UI, and section 3.8 is an optional post-setup verification.

| Step | What it does |
|---|---|
| 3.1 | Install NemoClaw (pinned `NEMOCLAW_INSTALL_REF`) and create the sandbox |
| 3.2 | Apply the VSS sandbox policy |
| 3.3 | Install the VSS skills |
| 3.4 | Upload the workspace bootstrap docs |
| 3.5 | Register the VSS Orchestrator MCP path *(HTTPS only; no-op when `ORCHESTRATOR_ENABLE_HTTPS` is `False`)* |
| 3.6 | Configure optional webhooks |
| 3.7 | Open the Agent UI |
| 3.8 | *(Optional)* Verify sandbox, policy, workspace, and webhooks |


### 3.1 Install NemoClaw and create the sandbox

Exports the NemoClaw configuration (including `CHAT_UI_URL`, the dashboard origin for UI access) and installs NemoClaw at the pinned `NEMOCLAW_INSTALL_REF`. The non-interactive installer also creates and onboards the sandbox. Takes a few minutes; output streams live.

When `NEMOCLAW_ENDPOINT_URL`'s host resolves to a non-public address, this step automatically starts the bundled host proxy and points NemoClaw at it, avoiding NemoClaw's SSRF rejection. Set the advanced setting `NEMOCLAW_INFERENCE_PROXY = False` to disable this behavior; it defaults to `True`.


In [ ]:
import ipaddress
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlsplit


if not isinstance(NEMOCLAW_INFERENCE_PROXY, bool):
    raise TypeError("NEMOCLAW_INFERENCE_PROXY must be True or False")


def _resolved_addresses(host):
    try:
        return sorted(
            {
                info[4][0]
                for info in socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            }
        )
    except socket.gaierror as exc:
        print(f"Could not resolve {host}; leaving the endpoint unchanged: {exc}", flush=True)
        return []


def _proxy_port_is_open():
    try:
        with socket.create_connection(("127.0.0.1", INFERENCE_API_PROXY_PORT), timeout=0.5):
            return True
    except OSError:
        return False


def _ensure_inference_api_proxy(upstream_host):
    if _proxy_port_is_open():
        print(f"Inference API proxy already listening on port {INFERENCE_API_PROXY_PORT}.", flush=True)
        return
    if not INFERENCE_API_PROXY_PATH.is_file():
        raise FileNotFoundError(
            f"NemoClaw needs the inference API proxy, but it was not found at {INFERENCE_API_PROXY_PATH}. "
            "Set INFERENCE_API_PROXY_PATH to inference-api-proxy.py."
        )

    proxy_env = os.environ.copy()
    proxy_env["INFERENCE_API_UPSTREAM"] = upstream_host
    proxy_env["INFERENCE_API_PROXY_HOST"] = "0.0.0.0"
    proxy_env["INFERENCE_API_PROXY_PORT"] = str(INFERENCE_API_PROXY_PORT)
    log_path = Path("/tmp/inference-api-proxy.log")
    with log_path.open("ab") as log_file:
        process = subprocess.Popen(
            [sys.executable, str(INFERENCE_API_PROXY_PATH)],
            env=proxy_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    for _ in range(50):
        if _proxy_port_is_open():
            print(
                f"Started {INFERENCE_API_PROXY_PATH} on port {INFERENCE_API_PROXY_PORT} "
                f"-> {upstream_host} (log: {log_path}).",
                flush=True,
            )
            return
        if process.poll() is not None:
            break
        time.sleep(0.1)
    raise RuntimeError(f"Inference API proxy failed to start; inspect {log_path}")


_effective_nemoclaw_endpoint_url = NEMOCLAW_ENDPOINT_URL
_inference_proxy_active = False
_endpoint = urlsplit(NEMOCLAW_ENDPOINT_URL) if NEMOCLAW_ENDPOINT_URL else None
_endpoint_host = _endpoint.hostname if _endpoint else None
if NEMOCLAW_INFERENCE_PROXY and _endpoint_host:
    _endpoint_addresses = _resolved_addresses(_endpoint_host)
    _has_non_public_address = any(
        not ipaddress.ip_address(address).is_global for address in _endpoint_addresses
    )
    if _has_non_public_address:
        print(
            f"{_endpoint_host} resolves to {_endpoint_addresses}; "
            "using the host proxy to avoid NemoClaw SSRF rejection.",
            flush=True,
        )
        _ensure_inference_api_proxy(_endpoint_host)
        _endpoint_path = _endpoint.path.rstrip("/") or "/v1"
        _effective_nemoclaw_endpoint_url = (
            f"http://host.openshell.internal:{INFERENCE_API_PROXY_PORT}{_endpoint_path}"
        )
        _inference_proxy_active = True

# Fail fast if no model provider is configured (section 1.2).
if NEMOCLAW_PROVIDER == "build" and not NVIDIA_API_KEY:
    raise RuntimeError(
        "No agent provider configured. Pick one and fill in its values:\n"
        "  - option (a) SOTA cloud / (b) local: set NEMOCLAW_ENDPOINT_URL, NEMOCLAW_MODEL, and COMPATIBLE_API_KEY in the (a) or (b) cell\n"
        "  - option (c) build.nvidia.com: set NVIDIA_API_KEY in the (c) cell"
    )
if NEMOCLAW_PROVIDER == "custom" and not COMPATIBLE_API_KEY:
    raise RuntimeError(
        "COMPATIBLE_API_KEY is required for the custom OpenAI-compatible provider (options a/b). "
        "Set it in the (a) or (b) cell (any non-empty placeholder works for most local servers)."
    )
if NEMOCLAW_PROVIDER == "custom" and not NEMOCLAW_MODEL:
    raise RuntimeError(
        "NEMOCLAW_MODEL is required when NEMOCLAW_ENDPOINT_URL is set. "
        "If you ran more than one of (a)/(b)/(c), re-run just the (a) or (b) cell you want to use."
    )

# NemoClaw reads its configuration from environment variables.
env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NEMOCLAW_INSTALL_REF"] = NEMOCLAW_INSTALL_REF
env["NEMOCLAW_PROVIDER"] = NEMOCLAW_PROVIDER
env["NEMOCLAW_NON_INTERACTIVE"] = "1"
env["NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE"] = "1"
env["NEMOCLAW_AGENT"] = AGENT_RUNTIME
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if NEMOCLAW_MODEL:
    env["NEMOCLAW_MODEL"] = NEMOCLAW_MODEL
if NEMOCLAW_PROVIDER == "custom":
    env["NEMOCLAW_ENDPOINT_URL"] = _effective_nemoclaw_endpoint_url
    env["COMPATIBLE_API_KEY"] = COMPATIBLE_API_KEY
    print(
        "Custom OpenAI-compatible provider: "
        f"{_effective_nemoclaw_endpoint_url} model={NEMOCLAW_MODEL} "
        f"key set={bool(COMPATIBLE_API_KEY)}"
    )
# CHAT_UI_URL must be correct at onboard time: the sandbox derives the UI
# allowed origins and the 0.0.0.0 forward bind from it (gateway.* is read-only later).
_chat_fqdn = brev_secure_link_fqdn(AGENT_DASHBOARD_PORT)
if _chat_fqdn:
    env["CHAT_UI_URL"] = f"https://{_chat_fqdn}"
    print("CHAT_UI_URL:", env["CHAT_UI_URL"])
else:
    print(
        f"WARNING: no FQDN for port {AGENT_DASHBOARD_PORT} in {BREV_ENVIRONMENT_CONTEXT_PATH}; "
        "onboard will proceed without baking a remote UI origin."
    )
os.environ.update(env)  # export for the ! commands below


def _version_of(text):
    match = re.search(r"v?(\d+\.\d+\.\d+)", text or "")
    return match.group(1) if match else None


install_ref = NEMOCLAW_INSTALL_REF.strip()
if not install_ref:
    raise RuntimeError("NEMOCLAW_INSTALL_REF must not be empty.")
install_url = f"https://raw.githubusercontent.com/NVIDIA/NemoClaw/{install_ref}/install.sh"


#### Install CLI and onboard the sandbox

Puts `~/.local/bin` on `PATH`, installs the pinned NemoClaw / NemoHermes CLI when the requested ref is not already present, then runs non-interactive onboard if the configured sandbox does not exist yet. Takes a few minutes on first run; safe to re-run.


In [ ]:
local_bin = HOME_DIR / ".local" / "bin"
if str(local_bin) not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = f"{local_bin}{os.pathsep}{os.environ.get('PATH', '')}"

installed = None
if shutil.which(AGENT_CLI):
    _ver = !{AGENT_CLI} --version 2>&1
    installed = _version_of(" ".join(_ver))

if installed and installed == _version_of(install_ref):
    print(f"{AGENT_LABEL} {installed} already installed, skipping installer.", flush=True)
else:
    print(f"Installing {AGENT_LABEL} {install_ref} (takes a few minutes)...", flush=True)
    !cd ~ && bash -o pipefail -c "curl -fsSL {install_url} | bash"
    install_exit_code = _exit_code
    if install_exit_code != 0:
        if not shutil.which(AGENT_CLI):
            raise AssertionError(f"{AGENT_LABEL} install failed")
        print(
            f"{AGENT_LABEL} installer exited non-zero after installing {AGENT_CLI}; "
            "continuing with explicit onboarding.",
            flush=True,
        )

!{AGENT_CLI} --version
assert _exit_code == 0, f"{AGENT_CLI} is not runnable after install"

!openshell sandbox get {NEMOCLAW_SANDBOX_NAME} >/dev/null 2>&1
if _exit_code != 0:
    print(
        f"No sandbox {NEMOCLAW_SANDBOX_NAME!r} yet — onboarding "
        f"(agent={AGENT_RUNTIME}, takes several minutes)...",
        flush=True,
    )
    !cd ~ && {AGENT_ONBOARD_CMD}
    if _exit_code != 0 and AGENT_RETRY_ONBOARD_ON_FAIL:
        print(f"{AGENT_LABEL} onboard failed; retrying with refreshed sandbox base-image resolution...", flush=True)
        os.environ.pop("NEMOCLAW_HERMES_SANDBOX_BASE_IMAGE_REF", None)
        os.environ["NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH"] = "1"
        !cd ~ && NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH=1 {AGENT_ONBOARD_CMD}
    if _exit_code != 0 and AGENT_RETRY_ONBOARD_ON_FAIL:
        print(f"{AGENT_LABEL} onboard still failed; retrying with --fresh and refreshed sandbox base-image resolution...", flush=True)
        !cd ~ && NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH=1 {AGENT_ONBOARD_FRESH_CMD}
    assert _exit_code == 0, f"{AGENT_CLI} onboard failed"
print(f"Sandbox {NEMOCLAW_SANDBOX_NAME!r} ready.", flush=True)


### 3.2 Apply the VSS sandbox policy

Merges `assets/vss_nemoclaw_policy.yaml` into the base OpenShell policy.


In [ ]:
if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")
_policy_add_cmd = AGENT_POLICY_ADD_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME, path=POLICY_PATH)
!{_policy_add_cmd}
assert _exit_code == 0, "policy add failed"


### 3.3 Install the VSS skills

Installs each `skills/<name>/` directory containing a `SKILL.md`. No gateway restart is needed: each install mirrors the skill into the agent's load path and resets the OpenClaw session index, so the next session discovers it.

The CLI installs one skill per call, so the cell runs `SKILL_INSTALL_WORKERS` installs at a time (set in 1.3; `1` installs serially). Output is buffered per skill.


In [ ]:
import concurrent.futures
import shlex
import subprocess

if not SKILLS_DIR.is_dir():
    raise FileNotFoundError(f"Missing skills dir: {SKILLS_DIR}")
skill_dirs = sorted({sp.parent for sp in SKILLS_DIR.glob("*/SKILL.md")})
if not skill_dirs:
    raise RuntimeError(f"No SKILL.md directories found under {SKILLS_DIR}")

# `skill install` takes one skill per call, and every call pays its own SSH
# handshake plus a round trip per file, so the set installs much faster spread
# across SKILL_INSTALL_WORKERS (set in 1.3). Each install writes to its own
# per-skill destination; the only shared writes are the active gateway selection
# and the OpenClaw session-index reset, and every worker writes the same value
# for this sandbox. A failed install aborts the cell; re-run it to retry.
_install_argv_template = shlex.split(AGENT_SKILL_INSTALL_CMD)


def _install_skill(skill_dir):
    # Format per token so a path containing spaces stays a single argv entry.
    argv = [t.format(sandbox=NEMOCLAW_SANDBOX_NAME, skill=str(skill_dir)) for t in _install_argv_template]
    return skill_dir, argv, subprocess.run(argv, capture_output=True, text=True)


def _report_install(argv, done):
    # Workers interleave, so buffer each install and emit it as one block.
    print("$", shlex.join(argv), flush=True)
    print((done.stdout or "") + (done.stderr or ""), end="", flush=True)


_install_failures = []
with concurrent.futures.ThreadPoolExecutor(max_workers=SKILL_INSTALL_WORKERS) as pool:
    # map() yields in submission order, so the log stays sorted by skill name.
    for skill_dir, argv, done in pool.map(_install_skill, skill_dirs):
        _report_install(argv, done)
        if done.returncode != 0:
            _install_failures.append(skill_dir)

assert not _install_failures, "skill install failed: " + ", ".join(d.name for d in _install_failures)
print(f"Installed {len(skill_dirs)} VSS skills ({SKILL_INSTALL_WORKERS} workers).", flush=True)


### 3.4 Upload the workspace bootstrap docs

Uploads the shared workspace `.md` docs, then the `_<variant>` overlay (overlay last, so it wins).


In [ ]:
if not WORKSPACE_DIR.is_dir():
    print(f"No workspace dir at {WORKSPACE_DIR}; skipping workspace upload.", flush=True)
else:
    docs = sorted(WORKSPACE_DIR.glob("*.md"))
    overlay_dir = WORKSPACE_DIR / f"_{WORKSPACE_VARIANT}"
    if overlay_dir.is_dir():
        docs += sorted(overlay_dir.glob("*.md"))
    # Self-heal: an earlier upload form used a file-path dest, which the
    # directory-semantics transport turned into <name>.md/<name>.md nesting on
    # fresh sandboxes. Remove any directory-shaped *.md leftovers first.
    !openshell sandbox exec -n {NEMOCLAW_SANDBOX_NAME} -- sh -c "mkdir -p {WORKSPACE_REMOTE_DIR} && find {WORKSPACE_REMOTE_DIR} -mindepth 1 -maxdepth 1 -type d -name '*.md' -exec rm -rf '{{}}' ';'"
    for doc in docs:
        # dest is a DIRECTORY: the OpenShell transport does mkdir + tar-extract
        # into it; a file path here collides with an existing file of that name.
        _upload_cmd = AGENT_UPLOAD_CMD.format(
            sandbox=NEMOCLAW_SANDBOX_NAME, doc=doc, dest=WORKSPACE_REMOTE_DIR
        )
        !{_upload_cmd} >/dev/null
        assert _exit_code == 0, f"upload failed: {doc.name}"
    print(f"Uploaded {len(docs)} workspace docs to {WORKSPACE_REMOTE_DIR}.", flush=True)


### 3.5 Register the VSS Orchestrator MCP

With `ORCHESTRATOR_ENABLE_HTTPS = False` (the default) this section is a no-op: the agent uses the locally deployed orchestrator MCP served by `deploy_vss_orchestrator.ipynb` (sections 3.3–4), so no registration is needed here. Set `ORCHESTRATOR_ENABLE_HTTPS = True` in both notebooks only if you want to register an HTTPS MCP endpoint with the sandbox instead.

When HTTPS is enabled, registers the host-side VSS Orchestrator MCP (`ORCHESTRATOR_MCP_SERVER`) at `ORCHESTRATOR_MCP_URL` — the `HOST_INTERNAL_ALIAS` address the sandbox can actually reach over `https`. Uses the selected harness MCP grammar (`AGENT_MCP_*_CMD`). Skipped if already registered, so if you flip the scheme after a first run, remove the old entry first (`AGENT_MCP_REMOVE_CMD` with that server name) before re-running.


In [ ]:
if not ORCHESTRATOR_ENABLE_HTTPS:
    print(
        "ORCHESTRATOR_ENABLE_HTTPS is False — skipping VSS Orchestrator MCP registration.",
        flush=True,
    )
else:
    _mcp_status_cmd = AGENT_MCP_STATUS_CMD.format(
        sandbox=NEMOCLAW_SANDBOX_NAME, server=ORCHESTRATOR_MCP_SERVER
    )
    !{_mcp_status_cmd} >/dev/null 2>&1
    if _exit_code == 0:
        print(f"MCP server {ORCHESTRATOR_MCP_SERVER!r} already registered.", flush=True)
    else:
        _mcp_add_cmd = AGENT_MCP_ADD_CMD.format(
            sandbox=NEMOCLAW_SANDBOX_NAME, server=ORCHESTRATOR_MCP_SERVER, url=ORCHESTRATOR_MCP_URL
        )
        !{_mcp_add_cmd}
        assert _exit_code == 0, "mcp add failed"


### 3.6 Configure optional agent webhooks

Writes harness-specific webhook config, then restarts the gateway so it takes effect. If the managed restart hits `SUPERVISOR_UNAVAILABLE`, falls back to `AGENT_RECOVER_CMD`. Skipped when `AGENT_HOOKS_ENABLED` is off.

- **OpenClaw** (`AGENT_RUNTIME=openclaw`): `hooks.enabled` / `hooks.path` / `hooks.token` on the dashboard port.
- **Hermes** (`AGENT_RUNTIME=hermes`): `platforms.webhook.*` in `/sandbox/.hermes/config.yaml` (port `AGENT_WEBHOOK_PORT`, default `8644`), then an OpenShell forward for that port. Hermes `gateway-token` is unrelated (API bearer on `:8642`).


In [ ]:
# Write webhook keys first, then restart separately. `config set --restart` can
# update the config and still exit non-zero with SUPERVISOR_UNAVAILABLE when the
# in-sandbox supervisor is gone; recover relaunches it.
# - OpenClaw: hooks.* (gateway.* is off-limits; UI origins come from CHAT_UI_URL).
# - Hermes: platforms.webhook.* (HMAC secret; listens on AGENT_WEBHOOK_PORT).
config_sets = []
if AGENT_HOOKS_ENABLED:
    if AGENT_RUNTIME == "openclaw":
        config_sets.append(("hooks.enabled", "true"))
        config_sets.append(("hooks.path", AGENT_HOOKS_PATH or "/hooks"))
        if AGENT_HOOKS_TOKEN:
            config_sets.append(("hooks.token", AGENT_HOOKS_TOKEN))
    elif AGENT_RUNTIME == "hermes":
        if not AGENT_HOOKS_TOKEN:
            raise RuntimeError("AGENT_HOOKS_TOKEN is required to configure Hermes webhooks.")
        config_sets.extend(
            [
                ("platforms.webhook.enabled", "true"),
                ("platforms.webhook.extra.port", str(AGENT_WEBHOOK_PORT)),
                ("platforms.webhook.extra.secret", AGENT_HOOKS_TOKEN),
                # Known route for callers; secret matches the global fallback.
                ("platforms.webhook.extra.routes.vss-notebook.secret", AGENT_HOOKS_TOKEN),
                ("platforms.webhook.extra.routes.vss-notebook.deliver", "log"),
                (
                    "platforms.webhook.extra.routes.vss-notebook.prompt",
                    "VSS notebook webhook: {__raw__}",
                ),
            ]
        )

if not config_sets:
    reason = (
        "webhooks disabled"
        if not AGENT_HOOKS_ENABLED
        else f"no webhook config for AGENT_RUNTIME={AGENT_RUNTIME!r}"
    )
    print(f"No sandbox config changes needed ({reason}).", flush=True)
else:
    for _key, _value in config_sets:
        _quoted = shlex.quote(_value)  # values may contain spaces/quotes
        _config_set_cmd = AGENT_CONFIG_SET_CMD.format(
            sandbox=NEMOCLAW_SANDBOX_NAME, key=_key, value=_quoted
        )
        !{_config_set_cmd}
        assert _exit_code == 0, f"config set failed: {_key}"
    print("Restarting gateway to apply webhook config...", flush=True)
    _gateway_restart_cmd = AGENT_GATEWAY_RESTART_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME)
    !{_gateway_restart_cmd}
    if _exit_code != 0:
        print("Managed gateway restart failed — falling back to sandbox recover...", flush=True)
        _recover_cmd = AGENT_RECOVER_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME)
        !{_recover_cmd}
        if _exit_code != 0:
            print("Recover reported failure — probing the gateway directly...", flush=True)
            assert agent_gateway_healthy(), "gateway is down after webhook config"
            print("Gateway is answering its health probe; continuing.", flush=True)
    print("Sandbox config applied; gateway restarted.", flush=True)

    if AGENT_RUNTIME == "hermes":
        # Manifest only forwards 18789/8642; webhook adapter needs its own forward.
        _wh_port = str(AGENT_WEBHOOK_PORT)
        !openshell forward stop {_wh_port} {NEMOCLAW_SANDBOX_NAME} >/dev/null 2>&1
        !openshell forward start --background {_wh_port} {NEMOCLAW_SANDBOX_NAME}
        assert _exit_code == 0, f"openshell forward start failed for webhook port {_wh_port}"
        print(
            f"Hermes webhook listening (forwarded): "
            f"http://127.0.0.1:{_wh_port}/health  "
            f"and routes at http://127.0.0.1:{_wh_port}/webhooks/<name> "
            f"(e.g. /webhooks/vss-notebook).",
            flush=True,
        )


### 3.7 Open the Agent UI

Open the **Agent UI** for the sandbox so you can chat with the agent and drive VSS from it.

- **OpenClaw** (`ui_uses_gateway_token`): prints the dashboard origin with a `#token=` fragment from `AGENT_GATEWAY_TOKEN_CMD`.
- **Hermes** (`dashboard_url_cmd`): prints `nemohermes <sandbox> dashboard-url` (plain URL; on Brev the host is rewritten to the secure-link FQDN). Hermes `gateway-token` is the OpenAI-compatible API bearer on port `8642`, not the UI token.

Run the next cell to check the dashboard forward, re-bind it if onboard's copy died (rebuild/recover) or is bound loopback-only, print a fresh **Agent UI** link, then open it in your browser. Only this sandbox's own forward is touched: the process holding the port must carry the sandbox id, so an unrelated service on `AGENT_DASHBOARD_PORT` stops the cell with an error instead of being adopted or killed.

<span style="color:red"><strong>Not on Brev?</strong> If you are accessing the UI from a different machine, open an SSH tunnel before opening the Agent web UI from below:<br/><code>ssh -L &lt;AGENT_DASHBOARD_PORT&gt;:127.0.0.1:&lt;AGENT_DASHBOARD_PORT&gt; &lt;user&gt;@&lt;agent-host&gt;</code><br/></span>


In [ ]:
import os
import re
import shlex
import subprocess
import time
from urllib.parse import urlparse, urlunparse


def fetch_gateway_token():
    if not AGENT_GATEWAY_TOKEN_CMD:
        raise RuntimeError(f"{AGENT_LABEL} has no gateway_token_cmd in AGENT_HARNESS_PROFILES")
    cmd = shlex.split(AGENT_GATEWAY_TOKEN_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME))
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout.strip()


def fetch_dashboard_url():
    if not AGENT_DASHBOARD_URL_CMD:
        raise RuntimeError(f"{AGENT_LABEL} has no dashboard_url_cmd in AGENT_HARNESS_PROFILES")
    cmd = shlex.split(AGENT_DASHBOARD_URL_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME))
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout.strip()


gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

_chat_fqdn = brev_secure_link_fqdn(AGENT_DASHBOARD_PORT)

# Onboard starts the dashboard forward, but it dies with the kernel or terminal that
# launched it and a rebuild/recover drops it; the UI then 503s. Re-establish it here.
_health = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}/health"


def _dashboard_forward_rows():
    """(BIND, PID, STATUS) rows OpenShell records for this sandbox's dashboard port."""
    listing = subprocess.run(["openshell", "forward", "list"], capture_output=True, text=True)
    rows = []
    for line in re.sub(r"\x1b\[[0-9;]*m", "", listing.stdout).splitlines():
        cols = line.split()
        if (
            len(cols) >= 5
            and cols[0] == NEMOCLAW_SANDBOX_NAME
            and cols[2] == str(AGENT_DASHBOARD_PORT)
        ):
            rows.append((cols[1], cols[3], cols[4]))
    return rows


def _port_listener_pids():
    """PIDs listening on the dashboard port; empty when lsof is unavailable."""
    try:
        listing = subprocess.run(
            ["lsof", "-t", f"-i:{AGENT_DASHBOARD_PORT}", "-sTCP:LISTEN"], capture_output=True, text=True
        )
    except FileNotFoundError:
        return []
    return listing.stdout.split()


def _pid_args(pid):
    """Command line of *pid*, empty when it is already gone."""
    return subprocess.run(["ps", "-p", pid, "-o", "args="], capture_output=True, text=True).stdout.strip()


def _dashboard_forward_holder():
    """(BIND, PID) of this sandbox's running forward that holds the port, or None."""
    # A row can read running after its process is gone and its PID be recycled, so the
    # listener's own command line, never the PID alone, decides whether it is ours.
    holders = set(_port_listener_pids())
    for bind, pid, status in _dashboard_forward_rows():
        if status == "running" and pid in holders and _is_sandbox_forward(_pid_args(pid)):
            return bind, pid
    return None


def _openshell_sandbox_describe(sandbox):
    """`openshell sandbox get` output, or raise when the host gateway cannot answer."""
    got = subprocess.run(["openshell", "sandbox", "get", sandbox], capture_output=True, text=True)
    if got.returncode != 0:
        detail = (got.stderr or got.stdout or "").strip() or f"exit {got.returncode}"
        raise RuntimeError(
            f"OpenShell cannot describe sandbox {sandbox!r}:\n{detail}\n"
            "This section needs the host-side OpenShell gateway; bring it back, then re-run."
        )
    return re.sub(r"\x1b\[[0-9;]*m", "", got.stdout)


def _openshell_sandbox_id(sandbox):
    """Sandbox id for *sandbox*, or None when the description carries no Id."""
    found = re.search(r"^\s*Id:\s+(\S+)", _openshell_sandbox_describe(sandbox), re.MULTILINE)
    return found.group(1) if found else None


_sandbox_id = _openshell_sandbox_id(NEMOCLAW_SANDBOX_NAME)


def _is_sandbox_forward(args):
    """True when a command line is this sandbox's forward. Without an id nothing is ours,
    so a foreign listener is never adopted and never killed."""
    return bool(_sandbox_id and re.search(rf"--sandbox-id[=\s]+{re.escape(_sandbox_id)}\b", args))

# A forward bound to 127.0.0.1 answers this health probe and still 503s behind the
# Brev secure link, which reaches the port from off-loopback, so the bind is checked
# too. 0.0.0.0 covers loopback, so it is never re-bound the other way. A healthy probe
# says nothing about who holds the port, so this sandbox must own a running forward.
_held = _dashboard_forward_holder()
_healthy = subprocess.run(["curl", "-fsS", "-o", "/dev/null", _health], capture_output=True).returncode == 0
# The port can change hands around the probe, so the same PID must still hold it after.
_bind = _held[0] if _held and _dashboard_forward_holder() == _held else None
if _healthy and _bind and (_bind == "0.0.0.0" or not _chat_fqdn):
    print(f"Dashboard forward already up on {_bind}:{AGENT_DASHBOARD_PORT}")
else:
    if _healthy and _bind:
        print(f"Dashboard forward is bound {_bind}, which the {_chat_fqdn} edge cannot reach; re-binding.")
    elif _healthy:
        print(f"Port {AGENT_DASHBOARD_PORT} answers but no running forward belongs to {NEMOCLAW_SANDBOX_NAME!r}; re-establishing.")
    # A dead row or an orphaned ssh child can still hold the port and make `forward
    # start` fail, so clear it first. Only this sandbox's own forward may be killed.
    subprocess.run(
        ["openshell", "forward", "stop", str(AGENT_DASHBOARD_PORT), NEMOCLAW_SANDBOX_NAME],
        check=False, capture_output=True, text=True,
    )
    for _pid in _port_listener_pids():
        _args = _pid_args(_pid)
        if not _is_sandbox_forward(_args):
            raise RuntimeError(
                f"Port {AGENT_DASHBOARD_PORT} is held by pid {_pid} ({_args}), which is not "
                f"{NEMOCLAW_SANDBOX_NAME!r}'s dashboard forward; stop it before continuing."
            )
        print(f"Clearing stale forward on {AGENT_DASHBOARD_PORT} (pid {_pid}): {_args}")
        subprocess.run(["kill", _pid])
    # setsid detaches the forward from this kernel's session so a kernel restart does
    # not take it down. A remote (non-loopback) UI origin needs 0.0.0.0, not 127.0.0.1.
    _fwd = f"0.0.0.0:{AGENT_DASHBOARD_PORT}" if _chat_fqdn else str(AGENT_DASHBOARD_PORT)
    !setsid -f openshell forward start --background {_fwd} {NEMOCLAW_SANDBOX_NAME}
    # setsid returns before the forward is up, and a healthy probe can come from an
    # unrelated listener, so readiness means one PID of ours holds the port across the
    # probe.
    for _ in range(15):
        _held = _dashboard_forward_holder()
        _answers = _held and subprocess.run(
            ["curl", "-fsS", "-o", "/dev/null", _health], capture_output=True
        ).returncode == 0
        if _answers and _dashboard_forward_holder() == _held:
            print("Dashboard forward ready on", _fwd)
            break
        time.sleep(1)
    else:
        print(f"WARNING: {_health} is not answered by {NEMOCLAW_SANDBOX_NAME!r}'s forward; check `nemoclaw {NEMOCLAW_SANDBOX_NAME} status`.")

if _chat_fqdn:
    origin = f"https://{_chat_fqdn}"
else:
    origin = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}"
    agent_host = subprocess.run(
        ["hostname", "-I"], capture_output=True, text=True, check=True
    ).stdout.split()[0]
    ssh_user = os.environ.get("USER", "ubuntu")
    RED, RESET = "\033[31m", "\033[0m"
    print(f"{RED}Make sure an SSH tunnel is running on your laptop before opening the Agent web UI:{RESET}")
    print(f"{RED}  $ ssh -L {AGENT_DASHBOARD_PORT}:localhost:{AGENT_DASHBOARD_PORT} {ssh_user}@{agent_host}{RESET}")
    if BREV_ENVIRONMENT_CONTEXT_PATH:
        print(
            f"{RED}No FQDN for port {AGENT_DASHBOARD_PORT} in {BREV_ENVIRONMENT_CONTEXT_PATH}; "
            f"using localhost tunnel URL instead.{RESET}"
        )

if AGENT_UI_USES_GATEWAY_TOKEN:
    # OpenClaw control UI authenticates via #token= from `sandbox gateway token`.
    token = fetch_gateway_token()
    agent_ui_url = f"{origin}/#token={token}" if token else origin
elif AGENT_DASHBOARD_URL_CMD:
    # Hermes: `dashboard-url` (plain URL). `gateway-token` is the :8642 API bearer — not for the UI.
    raw = fetch_dashboard_url()
    if _chat_fqdn and raw:
        parsed = urlparse(raw)
        agent_ui_url = urlunparse(
            ("https", _chat_fqdn, parsed.path or "", "", parsed.query, parsed.fragment)
        )
    else:
        agent_ui_url = raw or origin
else:
    agent_ui_url = origin

print("Agent UI:", agent_ui_url)
if AGENT_CONNECT_CMD:
    print(f"{AGENT_LABEL} terminal: {AGENT_CONNECT_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME)}")


### 3.8 [OPTIONAL] Verify sandbox, policy, workspace, and optional webhooks

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and version,
- whether agent webhooks are healthy when enabled (OpenClaw: `POST /hooks/agent`; Hermes: `GET :AGENT_WEBHOOK_PORT/health`),
- the installed OpenClaw skills/workspace files or Hermes top-level workspace docs.


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import subprocess
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
preset_name_match = re.search(r"^\s+name:\s*(\S+)", policy_text, re.MULTILINE)
print("Expected preset name:", preset_name_match.group(1) if preset_name_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not WORKSPACE_DIR.is_dir():
    raise FileNotFoundError(f"Missing plugin workspace source dir: {WORKSPACE_DIR}")
expected_workspace_md = tuple(sorted(p.name for p in WORKSPACE_DIR.glob("*.md")))
if not expected_workspace_md:
    raise RuntimeError(f"No .md files found in {WORKSPACE_DIR}; cannot verify workspace install.")
print("Expected workspace .md files:", list(expected_workspace_md))

if AGENT_HOOKS_ENABLED:
    if AGENT_RUNTIME == "openclaw":
        if not AGENT_HOOKS_TOKEN:
            raise RuntimeError("AGENT_HOOKS_TOKEN is required to verify OpenClaw hooks.")

        hooks_path = "/" + AGENT_HOOKS_PATH.strip("/")
        hooks_url = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}{hooks_path}/agent"
        hooks_payload = json.dumps(
            {
                "name": f"{AGENT_LABEL} notebook verification",
                "message": "test",
            }
        )
        hooks_cmd = [
            "curl",
            "-sS",
            "-w", "\n%{http_code}",  # append the HTTP status as the last stdout line
            "-X",
            "POST",
            hooks_url,
            "-H",
            f"Authorization: Bearer {AGENT_HOOKS_TOKEN}",
            "-H",
            "Content-Type: application/json",
            "-d",
            hooks_payload,
        ]
        hooks_result = subprocess.run(hooks_cmd, capture_output=True, text=True)
        hooks_body, _, hooks_status = (hooks_result.stdout or "").rpartition("\n")
        hooks_body, hooks_status = hooks_body.strip(), hooks_status.strip()
        if hooks_result.returncode != 0:
            # Only a transport failure exits non-zero here; 4xx still exits 0.
            print(
                f"{AGENT_LABEL} hooks test: FAIL — could not reach {hooks_url}"
                " — run section 3.7 to start the dashboard forward"
            )
        elif hooks_status == "200":
            print(f"{AGENT_LABEL} hooks test: PASS (HTTP 200)")
        else:
            hints = {
                "404": " — run section 3.6 to configure webhooks",
                "401": " — token mismatch; re-run section 3.6 to apply the current AGENT_HOOKS_TOKEN",
            }
            hint = hints.get(hooks_status, "")
            print(f"{AGENT_LABEL} hooks test: FAIL (HTTP {hooks_status or 'unknown'}){hint}")
    elif AGENT_RUNTIME == "hermes":
        health_url = f"http://127.0.0.1:{AGENT_WEBHOOK_PORT}/health"
        health = subprocess.run(["curl", "-fsS", health_url], capture_output=True, text=True)
        body = (health.stdout or "").strip()
        if health.returncode == 0 and '"status"' in body and "ok" in body:
            print(f"{AGENT_LABEL} webhook health: PASS ({health_url} -> {body})")
            print(f"  route example: http://127.0.0.1:{AGENT_WEBHOOK_PORT}/webhooks/vss-notebook")
        else:
            detail = body or (health.stderr or "").strip() or f"exit {health.returncode}"
            print(f"{AGENT_LABEL} webhook health: FAIL ({health_url}) — {detail}")
            print("  — run section 3.6 to configure Hermes webhooks / port forward")
    else:
        print(f"Agent hooks test skipped: no webhook verify for AGENT_RUNTIME={AGENT_RUNTIME!r}.")
else:
    print("Agent hooks test skipped: AGENT_HOOKS_ENABLED is false.")

if gateway_container:
    def _sandbox_exec(sandbox_cmd):
        return [
            "openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--",
            "sh", "-lc", sandbox_cmd,
        ]

    def _in_sandbox_show(label, sandbox_cmd):
        full = _sandbox_exec(sandbox_cmd)
        print(f"\n=== {label} ===")
        print("$", shlex.join(full))
        r = subprocess.run(full, capture_output=True, text=True, stdin=subprocess.DEVNULL)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
        return r

    if AGENT_VERIFY_KIND == "sandbox_docs":
        for _label_tmpl, _cmd_tmpl in AGENT_VERIFY_CMDS:
            _verify_fmt = dict(
                label=AGENT_LABEL,
                workspace_remote_dir=WORKSPACE_REMOTE_DIR,
                scheme=MCP_SCHEME,
                host_alias=HOST_INTERNAL_ALIAS,
                mcp_port=MCP_PORT,
            )
            _in_sandbox_show(
                _label_tmpl.format(**_verify_fmt),
                _cmd_tmpl.format(**_verify_fmt),
            )
    elif AGENT_VERIFY_KIND == "openclaw_workspace":
        # _in_sandbox_show("openclaw plugins list", "openclaw plugins list")
        _in_sandbox_show("openclaw plugins doctor", "openclaw plugins doctor")

        skills_proc = subprocess.run(
            _sandbox_exec("openclaw skills list --json"),
            capture_output=True, text=True, stdin=subprocess.DEVNULL,
        )
        workspace_dir = None
        print("\n=== openclaw skills (non-bundled) ===")
        try:
            payload = json.loads(skills_proc.stdout)
            skills = payload["skills"] if isinstance(payload, dict) else payload
            if isinstance(payload, dict):
                workspace_dir = payload.get("workspaceDir")
            non_bundled = [s for s in skills if not s.get("bundled", False)]
            if non_bundled:
                for s in non_bundled:
                    print(f"- {s['name']}")
            else:
                print("No non-bundled OpenClaw skills found.")
        except (json.JSONDecodeError, TypeError) as exc:
            print(f"Could not parse skills list JSON: {exc}")
            if skills_proc.stderr:
                print(skills_proc.stderr)

        print(f"\n=== workspace .md files in {workspace_dir or '<unknown>'} ===")
        if workspace_dir:
            ls_cmd = f"ls -1 {shlex.quote(workspace_dir)} 2>/dev/null"
            ls_proc = subprocess.run(_sandbox_exec(ls_cmd), capture_output=True, text=True, stdin=subprocess.DEVNULL)
            present = {line.strip() for line in ls_proc.stdout.splitlines() if line.strip().endswith(".md")}
            for name in expected_workspace_md:
                mark = "OK  " if name in present else "MISS"
                print(f"  {mark}  {name}")
            extra = present - set(expected_workspace_md)
            if extra:
                print(f"  (also present: {sorted(extra)})")
        else:
            print("  workspaceDir not found in skills JSON payload; check skipped.")
    else:
        print(f"No workspace verification handler for verify_kind={AGENT_VERIFY_KIND!r}.")
else:
    print("Could not determine the OpenShell gateway container; runtime-specific workspace checks were skipped.")
